<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP12A/MP12A_stability_estimation.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP12A · Stability Estimation after Contingencies

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Graph Neural Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem_12_2.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12_power_grid/")
PROJECT = (['mp12a.py', 'grid39c.py', 'MP12A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP12A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex12_power_grid", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex12_power_grid", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem_12_2', 'mp12a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem_12_2 as hs
import mp12a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP12A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex12.2's problem_12_2.py, by its name in Ex12.2
import grid39c as g                                      # the IEEE 39-bus grid with classical machines and converters

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

When a short circuit occurs on a transmission line, the protection opens the line after a fixed time, here **140 ms** (Ex12.2's). While the short circuit lasts, the machines near it cannot deliver their power and their rotors speed up; if it lasts too long, a machine loses synchronism with the others and must be disconnected. The longest short circuit the grid withstands is its **critical clearing time** (CCT). A transmission system operator checks it for every credible contingency at every operating point - **contingency screening** - and a case is **insecure** when its CCT is 140 ms or less: the protection would open the line too late.

Two things make this harder than in Ex12.2. The grid is larger: ten machines swing against each other, and which one loses synchronism depends on where the fault is. And **converters** - wind and solar plants connected through power electronics - replace machines. A converter delivers power but has no rotating mass: no inertia, no rotor angle. Where it replaces a machine, the remaining machines carry the swings alone, and the CCT of every contingency changes, unevenly.

The screen is repeated for every new operating point: every forecast, every quarter of an hour, every change of the converters' output. A simulation per contingency makes it slow; a network trained once can answer all the contingencies of a new operating point at once. **This mini project asks how accurately and how much faster** a graph neural network screens the 35 contingencies of the IEEE 39-bus grid with 0 to 60 % of its generation from converters.

## A2 · The grid, and what is new against Ex12.2

The grid is the **IEEE 39-bus system** (New England): 39 buses, 46 lines and transformers, 21 loads, 10 machines. Machine 1, at bus 39, stands for the rest of the interconnection (inertia constant 500 s), as the Nordic system does at bus 0 of Ex12.2's six-bus grid. The model is Ex12.2's, on this grid:

* the **network** is MATPOWER's `case39` (Athay, Podmore and Virmani, 1979): resistances, reactances, line charging and transformer taps, on a 100 MVA base; the grid is taken at **50 Hz** (case39 is a 60 Hz grid; assumed, as the course's grids are 50 Hz);
* the **machines are classical**: a constant voltage behind the transient reactance $x'_d$ and the swing equation, no voltage regulator, no turbine governor; $H$ and $x'_d$ of Pai (1989), checked against the IEEE 39-bus case shipped with ANDES; the **damping is assumed**, $D = H$;
* the **loads are constant impedances** at their pre-fault voltage (assumed, the usual choice of a classical stability study);
* a **contingency** is a solid three-phase fault at the first bus of a branch - a reactance of 1e-04 p.u. to earth - cleared by tripping that branch; the 35 branches whose loss leaves the grid in one piece are screened (the other 11 would island a machine or a load);
* the **operating points** scale every load by one factor between 0.85 and 1.15 of the base case, jitter each load by up to 10 %, and scale the machines' set points alike; the power flow is solved by Newton-Raphson.

**New against Ex12.2: converters.** At every operating point a share of the generation - **0, 20, 40 or 60 %** - comes from converters, placed at random among the nine local machines (machine 1 is never replaced). A converter injects the power the machine would have delivered, so the **power flow is unchanged**, but it has **no inertia**. It is modelled as a **negative constant impedance** at its bus: it delivers its pre-fault power at its pre-fault voltage and scales with the square of the voltage, so during a fault, where the voltage collapses, it delivers little, and it recovers with the voltage. This is assumed: it is the simplest model that keeps the network linear, and it does to first order what a current-limited converter does. It has no current limit, no fault ride-through control and no phase-locked loop; a real study would model those.

The cell below draws the grid with one operating point's converters and one contingency, and prints the machines.

In [ ]:
# The grid, one operating point's converters and one contingency, and the ten machines
D = np.load("MP12A_truth.npz")                            # the ground truth (part C explains how it was made)
i_op, j_ct = 195, 30                                      # an operating point with 60 % converters, and one contingency
g.draw_grid(D["converters"][i_op], g.CONTINGENCIES[j_ct],
            title=f"operating point {i_op}: {100*D['share_actual'][i_op]:.0f} % of the generation from converters; "
                  f"fault at bus {g.CONTINGENCIES[j_ct][0]+1}, branch {g.F[g.CONTINGENCIES[j_ct][1]]+1}-{g.T[g.CONTINGENCIES[j_ct][1]]+1} tripped")
plt.show()
op0 = g.power_flow()                                      # the base case
print("machine  bus   H [s]   x'_d [p.u.]   P at the base case [MW]")
for k in range(g.NM):
    print(f"{k+1:5d} {g.GEN_BUS[k]+1:5d} {g.H_GEN[k]:7.1f} {g.XDP[k]:10.3f} {100*op0['Pg'][k]:14.0f}")
print(f"load at the base case {100*op0['PL'].sum():.0f} MW; {g.N_BRANCH} branches, {g.N_CONT} contingencies screened")

**What you should see.** The 39 buses (the drawing places them by their distance along the branches, not by geography), the ten machines as circles and, at this operating point, seven of them replaced by converters (squares), 61 % of the generation; the fault marked by a star and the branch tripped dashed. The table: machine 1 at bus 39 with 500 s of inertia, the others 24 to 42 s; 6 254 MW of load at the base case.

## A3 · The equations and their conditions

**The swing equations** (Ex12.2, L12.2's *Transient Stability*). For every machine $i$ that is not a converter, the rotor angle $\delta_i$ [rad] and the speed deviation $\omega_i$ [rad/s] from the synchronous $\omega_s = 2\pi \cdot 50$ Hz follow

$$\frac{d\delta_i}{dt} = \omega_i, \qquad \frac{2H_i}{\omega_s}\frac{d\omega_i}{dt} = P_{m,i} - P_{e,i}(\boldsymbol{\delta}) - D_i\,\frac{\omega_i}{\omega_s}, \qquad
P_{e,i} = \sum_j E_i E_j \big(G_{ij}\cos\delta_{ij} + B_{ij}\sin\delta_{ij}\big)$$

with $E_i$ the internal voltage behind $x'_d$, $P_{m,i}$ the mechanical power [p.u.] and $G_{ij} + jB_{ij}$ the network **reduced onto the machines' internal nodes** (Kron reduction: loads, converters and the fault are impedances, so every bus without a machine can be eliminated exactly).

| when | condition | why it holds |
|---|---|---|
| before the fault | $\delta_i(0)$ the angles of the power flow, $\omega_i(0) = 0$, $P_{m,i} = P_{e,i}(\boldsymbol\delta(0))$ | the grid is in equilibrium at its operating point; the turbines' power does not change in 3 s (no governor) |
| fault on, $0 < t < t_c$ | the network with a reactance of $10^{-4}$ p.u. from the faulted bus to earth | a solid three-phase fault: the bus voltage collapses |
| after clearing, $t > t_c$ | the network without the tripped branch | the protection opens the branch at both ends |
| a converter | no angle, no speed: an impedance in the network | no rotating mass; its power follows its voltage |
| stability | stable if no two machines are more than 180° apart within 3 s of clearing | Ex12.2's criterion, beyond which a machine has slipped a pole |

**The critical clearing time** is the largest $t_c$ that is still stable, searched up to 500 ms; a case that is stable even at 500 ms is recorded as 500 ms ("at least 500 ms"), as in Ex12.2.

The cell below simulates one contingency cleared 10 ms before and 10 ms after its CCT.

In [ ]:
# One case cleared 10 ms before and 10 ms after its critical clearing time
op = mp.op_from_arrays(D, i_op); conv = D["converters"][i_op]
fb, br = g.CONTINGENCIES[j_ct]
cs = g.stack([g.case_networks(op, conv, fb, br)] * 2)
tc = float(D["cct"][i_op * g.N_CONT + j_ct])
_, t, d, w = g.simulate(cs, np.array([tc - 0.01, tc + 0.01]), 0.002, 3.0, record_dt=0.01)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
act = np.nonzero(~conv)[0]
for a, k, name in ((ax[0], 0, "cleared 10 ms before"), (ax[1], 1, "cleared 10 ms after")):
    for m in act[1:]:
        a.plot(t, np.degrees(d[k, :, m] - d[k, :, 0]), label=f"machine {m+1}")
    a.axhline(180, color="k", ls=":", lw=0.8)
    a.set_title(f"{name} the CCT ({1000*tc:.0f} ms)", fontsize=9); a.set_xlabel("time after clearing [s]")
ax[0].set_ylabel("rotor angle against machine 1 [deg]"); ax[0].set_ylim(-100, 400); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"CCT {1000*tc:.2f} ms; largest angle between two machines: {np.degrees(np.ptp(d[0][:, act], axis=1).max()):.0f} deg "
      f"cleared before it, {np.degrees(np.ptp(d[1][:, act], axis=1).max()):.0f} deg after it")

**What you should see.** This case's CCT is 77.17 ms. Cleared 10 ms before it (left), the machines swing apart - at most 95° - turn and come back together; cleared 10 ms after it (right), the machines near the fault swing on past 180° and lose synchronism with the rest (the angles run to thousands of degrees). Twenty milliseconds of fault decide between the two.

---

## A4 · Why there is no exact solution

The swing equations of several machines are nonlinear - the electrical power depends on the sines of the angle differences - and coupled through the network: they have no solution in closed form, and the CCT has no formula. The **equal area criterion** of L12.2 gives the CCT in closed form only for **one machine against an infinite bus**, without damping and with no electrical power during the fault: then the angle grows as a parabola in time while the fault lasts, and

$$t_c = \sqrt{\frac{4H\,(\delta_c - \delta_0)}{\omega_s P_m}}, \qquad \cos\delta_c = \frac{P_m(\delta_u - \delta_0)}{P_{\max}} + \cos\delta_u, \quad \delta_u = \pi - \arcsin\frac{P_m}{P_{\max}}$$

That case is the **check** of the solver in part C. For the 39-bus grid the ground truth is a converged simulation, checked three more ways: by refining its time step and its search, by the conservation of the transient energy in a lossless version, and against ANDES, an independent simulator.

## A5 · What is asked

| quantity | on | required |
|---|---|---|
| the CCT of every contingency | the 48 test operating points (1 680 cases), whose loads and converter placements you do not train on | mean error within **5 ms** of the ground truth |
| the verdict, insecure if the CCT is 140 ms or less | the same | **no insecure case missed**, at most **5 %** of the secure cases flagged insecure (false alarms) |
| the time to screen a new operating point (all 35 contingencies) | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline. Its online answer - the 35 CCTs of an operating point it has not seen - is timed against the reference model, which must simulate every contingency again. The training time is reported apart, as the offline cost.

**Why these numbers.** 5 ms is Ex12.2's agreed accuracy, under 4 % of the protection's 140 ms; the reference model meets it on average (part C). A missed insecure case is a blackout risk; a false alarm costs a simulation to dismiss it. A screen may therefore flag a case as insecure below a threshold slightly above 140 ms - a **margin** you choose - as long as it flags at most 5 % of the secure ones.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

As in Ex12.2, the network takes **one case** - one operating point and one contingency - as a **graph**, and returns its CCT:

$$\big(\hat A,\ H\big)\ \longmapsto\ t_c$$

* $\hat A$ (39 × 39): the grid **after** the branch is tripped, as an adjacency matrix with self-loops, normalised by the buses' degrees (Ex12.2's `normalised_adjacency`);
* $H$ (39 × $F$): $F$ numbers at every bus, known before the fault: the eight of the table below, scaled to zero mean and unit spread over the training cases;
* $t_c$: the CCT in units of 100 ms.

| feature | at every bus |
|---|---|
| $P$, $Q$ [p.u.] | the net active and reactive power injected, from the power flow |
| $\lvert V\rvert$ [p.u.], $\theta$ [rad] | the voltage magnitude and angle |
| is_machine | 1 where a machine swings |
| is_converter | 1 where a converter has replaced the machine |
| $H$ [100 s] | the inertia constant of the machine there, 0 for none or a converter |
| is_fault_bus | 1 at the faulted bus |

The contingency enters twice: the faulted bus is marked in $H$, and the tripped branch is missing from $\hat A$. The converters enter through `is_converter` and the inertia. **New against Ex12.2**: the converters, and a graph six times larger.

In [ ]:
# The data: 480 operating points x 35 contingencies
X, A, cct = D["X"], D["A"], D["cct"]
op_id, cont_id = D["op_id"], D["cont_id"]
train = op_id < int(D["n_train_ops"]); test = ~train
share = D["share"][op_id]                                # the converter share of each case
print(f"X {X.shape} (cases, buses, features); A {A.shape} (contingencies, buses, buses); cct {cct.shape}")
print("features:", ", ".join(D["feature_names"]))
print(f"training: {train.sum()} cases of {int(D['n_train_ops'])} operating points; test: {test.sum()} cases of {len(np.unique(op_id[test]))} operating points")
print("share   placed (mean) [%]   insecure [%]   CCT median [ms]   at 500 ms [%]   CCT 0 [%]")
for s in mp.SHARES:
    m = share == s
    print(f"{100*s:4.0f} %   {100*D['share_actual'][D['share'] == s].mean():12.1f}   {100*(cct[m] <= mp.PROTECTION_TIME).mean():12.1f}"
          f"   {1000*np.median(cct[m]):14.0f}   {100*(cct[m] >= mp.CCT_MAX).mean():12.1f}   {100*(cct[m] == 0).mean():8.1f}")

**What you should see.** 16 800 cases: 15 120 for training (432 operating points), 1 680 for the test (48 operating points). The converters' share is placed within a point of its target. Without converters 9.8 % of the cases are insecure; at 20 % 17.9 %, at 40 % 15.9 % and at 60 % 13.8 %. The share does not act in one direction: replacing a machine near a fault takes away a machine that would have accelerated (the CCT rises), replacing one far from it takes away inertia that held the others back (the CCT falls), and which happens depends on the placement. About 7 to 10 % of the cases are stable at 500 ms, and under 1 % lose synchronism even for the shortest fault (CCT 0): the grid without that branch cannot hold that operating point.

In [ ]:
# The CCT of each contingency, and how the converters change it
med = np.array([[np.median(cct[(cont_id == j) & (share == s)]) for j in range(g.N_CONT)] for s in mp.SHARES])
order = np.argsort(med[0])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k, s in enumerate(mp.SHARES):
    ax[0].plot(1000 * med[k, order], "o-", ms=3, lw=0.8, label=f"{100*s:.0f} % converters")
ax[0].axhline(140, color="k", ls=":", lw=0.8)
ax[0].set_xticks(range(g.N_CONT), [str(j) for j in order], fontsize=6)
ax[0].set_xlabel("contingency (sorted by its median CCT without converters)"); ax[0].set_ylabel("median CCT [ms]"); ax[0].legend(fontsize=7)
ax[1].hist([1000 * cct[share == s] for s in mp.SHARES], bins=25, label=[f"{100*s:.0f} %" for s in mp.SHARES])
ax[1].axvline(140, color="k", ls=":", lw=0.8); ax[1].set_xlabel("CCT [ms]"); ax[1].set_ylabel("cases"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
worst = order[:4]
for j in worst:
    fb, br = g.CONTINGENCIES[j]
    print(f"contingency {j:2d} (fault at bus {fb+1}, branch {g.F[br]+1}-{g.T[br]+1}): median CCT "
          + ", ".join(f"{1000*med[k, j]:.0f}" for k in range(4)) + " ms at 0/20/40/60 % converters")

**What you should see.** Left: the median CCT of each contingency, from the most critical to the least, for the four shares; the dotted line is the 140 ms of the protection. A handful of contingencies carry most of the risk - the faults at buses 28 and 26, on the branches 28-29, 26-29, 26-28 and 26-27, next to machine 9 (bus 38): median CCTs of 94, 112, 131 and 174 ms without converters - and the converters move every contingency's CCT, some up, some down; where machine 9 itself is replaced, the most critical of them is no longer critical (500 ms at 60 %). Right: the distribution of all the CCTs for each share, with the cases at the 500 ms ceiling at the far right.

## B2 · The physics the network is trained with

This network is **not** a PINN: like Ex12.2's, it learns from the ground truth's CCTs, and no equation appears in its loss. The physics enters in three other places (L12.2's *Three Places to Put the Physics*):

| where | what it carries |
|---|---|
| the graph $\hat A$ | which buses are connected after the trip: a graph layer passes information along the branches, one branch per layer |
| the bus features | where the machines and converters are, how much inertia each machine has, how loaded the grid is, where the fault is |
| the labels | the swing equations, solved by the ground truth |

**What more can be built in.** Four numbers per machine are cheap to compute from the power flow and decide much of the first swing: the **acceleration of each machine during the fault**, $(P_{m,i} - P_{e,i}^{\mathrm{fault}}(\boldsymbol\delta_0))\,\omega_s/2H_i$, the **imbalance just after clearing** at the pre-fault angles, $(P_{m,i} - P_{e,i}^{\mathrm{post}}(\boldsymbol\delta_0))\,\omega_s/2H_i$, the machine's **power per inertia**, $P_{m,i}/2H_i$, and its **synchronising coefficient** after clearing, $\partial P_{e,i}/\partial\delta_i \cdot \omega_s/2H_i$, the square of the angular frequency at which it would swing alone against the others. They need the fault-on and post-fault reduced networks, two Kron reductions per contingency (a few milliseconds for all 35), not a simulation. `mp.physics_features(op, conv)` computes them for one operating point (35 × 39 × 4), and `mp.physics_features_all(D)` for every case of the data. Whether you use them is your choice in part D; the cell below shows how they relate to the CCT.

**The loss** is the mean squared difference between the predicted and the true CCT, with one correction: a case recorded at 500 ms is stable **at least** that long, so a prediction above 500 ms is not an error there (a **censored** label).

In [ ]:
# The physics features, and how the fastest machine's acceleration relates to the CCT
t0 = time.perf_counter()
XP = mp.physics_features_all(D)                          # (16800, 39, 4): fault-on acceleration, imbalance after clearing, P_m / 2H, synchronising coefficient
print(f"physics features of all {len(XP)} cases in {time.perf_counter() - t0:.1f} s ({1000*(time.perf_counter() - t0)/len(D['share']):.0f} ms per operating point)")
a_max = XP[:, :, 0].max(1) * 100                          # the largest acceleration during the fault [rad/s²]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
sc = ax.scatter(a_max, 1000 * cct, s=2, c=share, cmap="viridis", alpha=0.6)
ax.axhline(140, color="k", ls=":", lw=0.8)
ax.set_xlabel("largest acceleration of a machine during the fault [rad/s²]"); ax.set_ylabel("CCT [ms]")
plt.colorbar(sc, ax=ax, label="converter share"); plt.tight_layout(); plt.show()
inside = (cct > 0) & (cct < mp.CCT_MAX)
print(f"correlation of 1/sqrt(largest acceleration) with the CCT (cases below 500 ms): "
      f"{np.corrcoef(1 / np.sqrt(a_max[inside]), cct[inside])[0, 1]:.2f}")

**What you should see.** The four features of all 16 800 cases in under half a minute (a few tens of ms per operating point). The CCT falls as the fastest machine accelerates harder during the fault, roughly as one over the square root of the acceleration - the parabola of section A4; the correlation is 0.77 - but with a wide spread: the same acceleration can give 100 or 300 ms, because the CCT also depends on how strongly the grid pulls the machine back after clearing, and on which other machines swing with it. That spread is what the network must learn.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | what an engineer would run for a new operating point | the same method, refined until it no longer changes |
| time step of RK4 | 10 ms (half a cycle at 50 Hz, the usual step of a transient stability program) | 2 ms |
| search for the CCT | bisection from [0, 500 ms] to an interval of 7.8 ms (six halvings) | bisection to 0.01 ms (sixteen halvings) |
| its job | to be **timed** against the network: the online cost of screening a new operating point | to **score** the network and the reference model |

Both run the same code, `g.cct`, on the same reduced networks. This part explains the scheme, builds confidence in the ground truth, and measures the reference model's accuracy and time: the bar part D has to clear.

## C2 · The scheme, step by step

1. **The power flow** of the operating point (Newton-Raphson; `g.power_flow`): the voltage and angle at every bus, the machines' power. It is shared by the network and the reference model, so it is not timed.
2. **The internal voltages** $E_i e^{j\delta_i(0)} = V_i + jx'_{d,i}\,(P_i - jQ_i)/V_i^*$ of the machines that swing.
3. **Three reduced networks** per contingency, before, during and after the fault (`g.case_networks`): loads and converters as impedances at their pre-fault voltage, the fault as a shunt, and every bus eliminated (Kron reduction), leaving a 10 × 10 matrix with zero rows for the converters.
4. **RK4 in time** (`g.simulate`), for all 35 contingencies at once: the fault-on interval is divided into equal steps of at most the time step, so the fault is **cleared exactly at $t_c$**, then 3 s at the time step on the post-fault network. The largest angle between two machines that swing is kept.
5. **Bisection on $t_c$** (`g.cct`): a fault of 500 ms that is still stable gives 500 ms; one of a tenth of the tolerance that is not gives 0; otherwise the interval [0, 500 ms] is halved, keeping the half where the verdict changes, until it is narrower than the tolerance; the CCT is its midpoint.

Each halving costs one simulation of every contingency; the reference model needs eight simulations, the ground truth eighteen.

In [ ]:
# The ground truth's convergence, from the truth file: operating point 0 (no converters) and 3 (60 %), 70 cases
steps, tols = D["conv_steps"], D["conv_tols"]
S, Tt = D["conv_step_cct"], D["conv_tol_cct"]
inside = (S[-1] > 0) & (S[-1] < mp.CCT_MAX)
print("RK4 step [ms]       largest change of the CCT [ms]   (bisected to 1e-04 ms)")
for k in range(1, len(steps)):
    print(f"{1000*steps[k-1]:5.0f} -> {1000*steps[k]:3.0f}          {1000*np.abs(S[k] - S[k-1])[inside].max():.2e}")
print("bisection to [ms]   largest change of the CCT [ms]   (RK4 on 2 ms)")
for k in range(1, len(tols)):
    print(f"{1000*tols[k-1]:6.3f} -> {1000*tols[k]:6.3f}    {1000*np.abs(Tt[k] - Tt[k-1])[inside].max():.2e}")
print(f"the ground truth: 2 ms, bisection to {1000*float(D['truth_tol']):.2f} ms; {1000*float(D['truth_seconds_per_case']):.0f} ms of CPU a case")

**What you should see.** Halving the time step from 40 ms changes no CCT by more than 0.013 ms, and the change falls about tenfold per halving to near the bisection's own resolution of 1e-04 ms: RK4 is a fourth-order method, and the swings it integrates last about a second, so even a 10 ms step resolves them. **The time step is not what limits the accuracy; the search is.** Each tenfold refinement of the bisection changes the CCT by about the tolerance: 0.46 ms, then 0.03, then 0.003. The ground truth, 2 ms and 0.01 ms, is therefore within about 0.01 ms of the converged CCT - five hundred times finer than the 5 ms asked.

In [ ]:
# Two checks run here: the exact one-machine case, and the conservation of the transient energy
print("one machine against an infinite bus, no damping, no fault power (section A4):")
g.smib_check()
print("the transient energy after clearing, with the network made lossless and the damping removed:")
for i, j in ((0, 3), (3, 20)):
    g.energy_check(mp.op_from_arrays(D, i), D["converters"][i], *g.CONTINGENCIES[j])

**What you should see.** For three loadings of the one-machine case, the course's RK4 and bisection return the equal area criterion's CCT within 0.02 ms: the code is right where the answer is known. With the conductances removed and no damping, the transient energy - kinetic energy of the rotors plus the potential energy of the network - must stay constant after clearing; on the 39-bus grid, with and without converters, it changes by about 1e-10 of the kinetic energy gained during the fault over 3 s. The integrator conserves what the physics conserves.

## C3 · An independent check: ANDES

Ex12.2 section 5 compared its simulation with **ANDES** (Cui, Li and Tomsovic, *IEEE Trans. Power Systems*, 2021), an open-source power system simulator. Here ANDES was given the same network, the same operating points, the same classical machines (its GENCLS model), loads and converters as constant impedances, and found the CCT by the same bisection, to 1 ms, with its own implicit trapezoidal method on 2 ms - about 13 s per CCT. It was run on 24 test cases chosen at random when the truth file was made (it is not needed here); the results are in the truth file. ANDES writes a solid fault as a reactance to earth; the ground truth uses the same, 1e-04 p.u. (Ex12.2 used a resistance of 1e-04 p.u.; with it the CCTs differ by about 1 ms).

In [ ]:
# ANDES against the ground truth, from the truth file
pk, ca, ct = D["andes_picks"], D["andes_cct"], D["andes_truth"]
print("operating point  share  contingency   ANDES [ms]   ground truth [ms]   difference [ms]")
for (i, j), a, t, m in zip(pk, ca, ct, D["andes_milder"]):
    print(f"{i:10d} {100*D['share'][i]:8.0f} % {j:8d}   " + ("  no answer" if np.isnan(a) else f"{1000*a:9.1f}") +
          f"   {1000*t:12.1f}      " + ("" if np.isnan(a) else f"{1000*(a - t):+8.1f}") + ("   milder fault in a run" if m else ""))
ok = ~np.isnan(ca); same = ok & ~D["andes_milder"]; dd = np.abs(ca - ct)
print(f"\nANDES answered {ok.sum()} of {len(ca)}; with the same fault in every run ({same.sum()} cases) within {1000*dd[same].max():.2f} ms, "
      f"median {1000*np.median(dd[same]):.2f} ms; with a milder fault in a run ({(ok & ~same).sum()}) {1000*dd[ok & ~same].min():.1f} to {1000*dd[ok & ~same].max():.1f} ms; "
      f"its power flow agrees to {D['andes_power_flow'][:, 0].max():.1e} p.u. and {D['andes_power_flow'][:, 1].max():.1e} deg; "
      f"{np.mean(D['andes_seconds']):.0f} s per CCT")

**What you should see.** ANDES's power flow agrees with the course's to 3e-05 degrees. ANDES answered 20 of the 24 cases. In the 13 whose every run used the same fault as the ground truth, a reactance of 1e-04 p.u., **ANDES's CCT is within 0.5 ms of the ground truth** (median 0.25 ms), at every converter share: the same model in two simulators. In the other seven ANDES could not converge just after clearing in some run of its bisection - the fault removed and the branch tripped at the same instant - and repeated it with a milder fault of 1e-03 p.u.; those differ by 0.2 to 11 ms, in the direction a milder fault gives (a longer CCT). In four cases it could not finish a run at all. ANDES needed about half a minute for one CCT, the course's code about 0.1 s.

## C4 · The reference model: accuracy and time

The reference model is the coarsest that meets the agreed 5 ms on average: RK4 on 10 ms and six halvings, a CCT within 3.9 ms of the truth (the midpoint of a 7.8 ms interval). Five halvings would leave a 15.6 ms interval and errors up to 7.8 ms. The truth file holds its CCT for all 8 400 cases; the cell below scores it on the test cases, then runs it on three test operating points on this runtime - the same numbers as the file, and the online time of a new operating point.

In [ ]:
# The reference model against the ground truth on the 1680 test cases, and its time for a new operating point
ref = D["cct_ref"]
err = np.abs(ref - cct)[test]
c_ref = pb.confusion(cct[test], ref[test])               # positive: insecure (CCT of 140 ms or less)
print(f"reference model on {test.sum()} test cases: mean error {1000*err.mean():.2f} ms, worst {1000*err.max():.2f} ms; "
      f"missed {c_ref['missed']} of {c_ref['tp'] + c_ref['fn']} insecure, false alarms {c_ref['alarms']} "
      f"({100*c_ref['alarms']/(c_ref['fp'] + c_ref['tn']):.2f} % of the secure)")
times = []
for i in (432, 453, 479):                                 # three test operating points
    op, conv = mp.op_from_arrays(D, i), D["converters"][i]
    c = mp.reference(op, conv)
    same = np.abs(c - ref[op_id == i]).max()
    times.append(middle(lambda: mp.reference(op, conv), reps=3))
    print(f"operating point {i} ({100*D['share'][i]:.0f} % converters): 35 CCTs in {times[-1]:.2f} s; "
          f"largest difference from the file {1000*same:.1e} ms")
t_ref = float(np.median(times))
print(f"\nthe network must screen a new operating point in under {1000*t_ref/10:.0f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "mean_err_ref": float(err.mean()), "worst_ref": float(err.max()),
       "missed_ref": c_ref["missed"], "alarms_ref": c_ref["alarms"]}          # kept for part E

**What you should see.** On the test cases the reference model is within 1.75 ms of the ground truth on average and 3.90 ms at worst, inside the 5 ms agreed; at 140 ms it misses none of the 197 insecure cases and flags 2 of the 1 483 secure ones. It reproduces the file's CCTs exactly, and screens one operating point - 35 contingencies, eight simulations of each - in 2.5 to 2.7 s on the authors' CPU, shared with other jobs (about 1 s alone). That sets the bar: a tenth of it for the network, on the same runtime.

---

# Part D · The Graph Neural Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** with the screen's **margin** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `net(A, H)`:

| argument | meaning | shape |
|---|---|---|
| `A` | the normalised adjacency of the grid after the trip, each branch weighted by its susceptance $1/x$ | `(n, 39, 39)` |
| `H` | the numbers at the buses, scaled: the 8 of section B1 then the 4 physics features of section B2 | `(n, 39, 12)` |
| returns | the CCT in units of 100 ms (0 to 5 for 0 to 500 ms) | `(n,)` |

Whatever you compute inside - graph layers, pooling, which features you use - stays inside `net`. The loss, the score and the evaluation only call `net(A, H)`. `MARGIN` [s] is your screen's margin: a case is flagged insecure if the predicted CCT is at most 140 ms + `MARGIN`.

## D2 · The data and the loss, given

The cell below builds the inputs of every case: the 35 adjacency matrices weighted by the branches' susceptance and normalised as in Ex12.2 (`pb.normalised_adjacency`; `mp.weighted_adjacency` gives the weights), the 12 numbers per bus scaled to zero mean and unit spread over the training cases, and the labels in units of 100 ms. `inputs(idx)` returns the tensors of the cases `idx`. `loss_fn(pred, y)` is the mean squared error with the censored label of section B2: where the truth is 500 ms, a prediction above it is not an error. `quick_score()` gives the mean error on the test cases [ms] and the screen's missed cases and false alarms at your `MARGIN`, in a second.

In [ ]:
# The data and the loss, given -------------------------------------------------------
F_all = np.concatenate([X, XP], -1)                       # (16800, 39, 12): the 8 bus numbers, then the 4 physics features
mu, sd = F_all[train].reshape(-1, F_all.shape[-1]).mean(0), F_all[train].reshape(-1, F_all.shape[-1]).std(0)
sd = np.where(sd > 1e-12, sd, 1.0)
H_all = torch.tensor((F_all - mu) / sd)                   # scaled over the training cases
A_all = torch.tensor(pb.normalised_adjacency(A * mp.weighted_adjacency()[None]))   # (35, 39, 39), one per contingency
y_all = torch.tensor(cct / 0.1)                           # the labels, in units of 100 ms
cid = torch.tensor(cont_id)
train_idx, test_idx = np.nonzero(train)[0], np.nonzero(test)[0]

def inputs(idx):
    """The graphs and the bus numbers of the cases idx, on the device."""
    return A_all[cid[idx]].to(DEVICE), H_all[idx].to(DEVICE)   # onto the GPU when there is one

def loss_fn(pred, y):
    r = pred - y
    r = torch.where(y >= 5.0 - 1e-9, torch.clamp(r, max=0.0), r)   # censored at 500 ms: only a prediction below is an error
    return r.pow(2).mean()

def predict(idx, chunk=2000):
    """The network's CCT [s] of the cases idx."""
    net.eval()                                            # evaluation mode
    with torch.no_grad():                                 # evaluation only: no gradients
        out = torch.cat([net(*inputs(b)) for b in np.array_split(idx, max(1, len(idx) // chunk))])
    net.train()                                           # back to training mode
    return np.clip(out.cpu().numpy(), 0.0, 5.0) * 0.1

MARGIN = 0.0                                              # your screen's margin [s]; set it in TODO 2
def quick_score():
    p, t = predict(test_idx), cct[test_idx]
    e = np.where((t >= mp.CCT_MAX) & (p >= mp.CCT_MAX), 0.0, np.abs(p - t))
    c = pb.confusion(t, p - MARGIN)
    return f"mean error {1000*e.mean():.2f} ms, missed {c['missed']} of {c['tp'] + c['fn']}, false alarms {100*c['alarms']/(c['fp'] + c['tn']):.1f} %"

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **the graph layers.** Ex12.2's layer, $H' = \tanh(H W_\mathrm{self} + b + \hat A H W_\mathrm{neigh})$, passes information one branch per layer. The 39-bus grid is up to ten branches across, against Ex12.2's three: how many layers, how wide, with residual connections or not?
* **the readout.** Ex12.2 took the mean over the buses. Here the fault is at one bus and ten machines may slip: the mean, the maximum, the faulted bus's own numbers, the machines' numbers?
* **which inputs.** All 12 numbers, or only the 8 of Ex12.2's kind (`H[..., :8]`): section B2 suggests the physics features carry much of the first swing.

What the data tell you: CCTs from 0 to 500 ms, 15 % of them insecure, decided by a handful of contingencies near machines 8 and 9; converters move them up or down by tens of milliseconds.

In [ ]:
# TODO 1 - your network. net(A, H) -> CCT in units of 100 ms, shape (n,)
import torch.nn as nn

class Net(nn.Module):                                     # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, A, H):
        ...                                              # return the CCT / 100 ms, shape (n,)

torch.manual_seed(88)                                     # fix the random start so a run repeats
net = Net().to(DEVICE)                                    # the network, on the GPU when there is one
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training, and your margin

Train `net` with `loss_fn` on the training cases (`train_idx`, `inputs`). Decide the optimiser, the number of steps, the learning rate and its schedule, the batch size. Keep the loss history for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

Then choose `MARGIN` [s], **on the training cases only**: the smallest margin that misses no insecure training case, or a little more. The test cases score it. `quick_score()` prints the test score at any time.

In [ ]:
# TODO 2 - your training, and your margin
history = []
t0 = time.perf_counter()
...                                                      # your optimiser and loop; append the loss to history
TRAIN_SECONDS = time.perf_counter() - t0
MARGIN = ...                                             # [s], chosen on the training cases
print(f"trained in {TRAIN_SECONDS:.0f} s; margin {1000*MARGIN:.1f} ms; test: {quick_score()}")

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss during training, and the predicted against the true CCT of the test cases, with the 140 ms line and your margin. Save each figure you want in the report as `MP12A_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the CCT of the 1 680 test cases, and the screen's verdicts at your `MARGIN`;
* the CCT of the test cases of each converter share apart;
* the **online time** of a new operating point on this runtime: everything the network needs after the power flow - the 35 contingencies' physics features (two Kron reductions each), the bus numbers, the scaling, and one call of `net` for all 35 - against the reference model on the same operating point, on three test operating points; and the network's time for all 48 test operating points at once.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()                                                # evaluation mode
R = {"cct_test": predict(test_idx), "cct_true": cct[test_idx], "share_test": share[test_idx],
     "cont_test": cont_id[test_idx], "margin": MARGIN}

def screen(op, conv):
    """The network's 35 CCTs [s] of a new operating point: what it computes after the power flow."""
    F = np.concatenate([np.stack([mp.features(op, conv, fb) for fb, _ in g.CONTINGENCIES]), mp.physics_features(op, conv)], -1)
    with torch.no_grad():                                 # evaluation only: no gradients
        out = net(A_all.to(DEVICE), torch.tensor((F - mu) / sd, device=DEVICE))   # the 35 graphs at once, on the GPU when there is one
    return out.cpu().numpy() * 0.1

t_net, t_ref = [], []
for i in (432, 453, 479):                                 # three test operating points
    op, conv = mp.op_from_arrays(D, i), D["converters"][i]
    assert np.abs(np.clip(screen(op, conv), 0, 0.5) - R["cct_test"][op_id[test_idx] == i]).max() < 1e-6
    t_net.append(middle(lambda: screen(op, conv)))
    t_ref.append(middle(lambda: mp.reference(op, conv), reps=3))
R["net_1"], R["ref_1"] = float(np.median(t_net)), float(np.median(t_ref))
R["net_48"] = middle(lambda: predict(test_idx))           # all 1680 test cases at once (inputs already built)
R["train_seconds"] = TRAIN_SECONDS
R["n_weights"] = sum(p.numel() for p in net.parameters())   # number of trainable weights and biases
R["device"] = DEVICE
torch.save(net.state_dict(), cc.output_path("MP12A_net.pt"))   # the trained weights, beside the report
print(f"online, one new operating point (35 contingencies): network {1000*R['net_1']:.1f} ms, reference {1000*R['ref_1']:.0f} ms "
      f"({R['ref_1']/R['net_1']:.0f} times)")
print(f"the network on all 48 test operating points at once: {1000*R['net_48']:.1f} ms; training {TRAIN_SECONDS:.0f} s offline")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `D` and the reference model's results `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
print("in R:", ", ".join(R))
print(f"reference model (part C): mean error {1000*REF['mean_err_ref']:.2f} ms, worst {1000*REF['worst_ref']:.2f} ms, "
      f"{REF['t_ref_s']:.2f} s per new operating point")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the mean, root-mean-square and worst error of the CCT on the test cases [ms] (a case at 500 ms predicted above 500 ms has no error);
* the insecure test cases missed, and the false alarms as a percentage of the secure test cases, at your margin;
* the mean error at each converter share [ms];
* the network's online time for a new operating point, the reference model's on the same runtime, and their ratio;
* the offline training time.

Then say in one line each whether the three requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>55}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP12A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the predicted against the true CCT of the test cases, with 140 ms and your margin; the error by contingency or by converter share; and the online times of the network and the reference model.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 7 are about your network and its result, 8 and 9 ask you to use it, 10 concludes. The next cell writes `MP12A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its graph layers (how many, how wide, residual or not), its readout, the inputs it uses, and its number of weights. Why this design for a grid of 39 buses and ten machines?": """
""",
    "2. What did the physics features of section B2 and the susceptance weights of the graph change? Give the mean test error and the missed cases of your network with and without them (one training each), and say why.": """
""",
    "3. Your training: optimiser, steps, batch size, learning rate and schedule, and the offline time on the T4 in seconds. How did you choose the margin on the training cases, and what is it in ms?": """
""",
    "4. The CCT on the 1 680 test cases: your mean, root-mean-square and worst errors in ms, against the reference model's 1.75 ms mean and 3.90 ms worst. Is the 5 ms requirement met? For which contingencies is your error largest, and why there?": """
""",
    "5. The screen at your margin: insecure test cases missed and false alarms in per cent of the secure ones, against the reference model's. Draw or tabulate how both change with the margin from 0 to 30 ms.": """
""",
    "6. The converter shares: your mean error, missed cases and false alarms at 0, 20, 40 and 60 % converters. Which share is hardest for the network, and what in the physics of section B makes it so?": """
""",
    "7. The online time of a new operating point: your network against the reference model on the same runtime, in ms, and the ratio; and the network for all 48 test operating points at once. What does each method spend its time on?": """
""",
    "8. Take the test operating point 479 (60 % converters) and its most critical contingency. Compute its true CCT with the converters replaced by their machines (mp.ground_truth with no converters) and with them; does your network predict the change in sign and size?": """
""",
    "9. Train again with another seed. How much do the mean error, the missed cases and the false alarms of the two runs differ? Is your screen reproducible within the requirements?": """
""",
    "10. To conclude: would you put this screen in front of an operator, and for what? Say what it may decide alone, what it must pass to the simulation, and what you would check before trusting it on a converter share above 60 %.": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP12A — Stability Estimation after Contingencies", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP12A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP12A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP12A_report.md into MP12A_report.pdf, with any figure
# saved as MP12A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP12A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP12A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP12A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP12A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass